# Latency Benchmarks
### End-to-end latency analysis for all system components
---

In [ ]:
import sys, os, time
sys.path.append('..')
import numpy as np
import matplotlib.pyplot as plt
from src.qkd.bb84        import BB84, RANChannel
from src.pqc.kyber       import KyberKEM
from src.hybrid.combiner import HybridKeyCombiner
from src.hybrid.dual_encrypt import IoTPacketEncryptor
from config import *
import warnings
warnings.filterwarnings('ignore')
print('Setup complete')

## 1. Component latency — 100 trials each

In [ ]:
TRIALS  = 100
channel = RANChannel(RAN_NOISE_NORMAL, RAN_LOSS_NORMAL)

# Pre-generate keys
qkd_r  = BB84(QKD_NUM_QUBITS, channel).run()
kem_r  = KyberKEM(PQC_ALGORITHM).full_key_exchange()
comb   = HybridKeyCombiner(HYBRID_METHOD)
hybrid = comb.combine(qkd_r['secret_key'], kem_r['shared_secret'])

def time_it(fn, trials=TRIALS):
    times = []
    for _ in range(trials):
        t = time.perf_counter()
        fn()
        times.append((time.perf_counter() - t) * 1000)
    return times

print('Measuring components...')
t_bb84   = time_it(lambda: BB84(QKD_NUM_QUBITS, channel).run())
t_kyber  = time_it(lambda: KyberKEM(PQC_ALGORITHM).full_key_exchange())
t_hkdf   = time_it(lambda: comb.combine(qkd_r['secret_key'], kem_r['shared_secret']))

enc = IoTPacketEncryptor(
    hybrid['hybrid_key'],
    qkd_key=qkd_r['secret_key'],
    kyber_key=kem_r['shared_secret'],
    mode='DUAL'
)
t_enc = time_it(lambda: enc.decrypt_packet(
    enc.encrypt_packet('hospital', id='003', bp='120/80', hr=72, crit='NO', ts=1711360800)
))

names  = ['BB84 QKD', 'Kyber768', 'HKDF', 'AES Dual']
timess = [t_bb84, t_kyber, t_hkdf, t_enc]

print(f'\n  {"Component":<15} {"Avg":>8} {"Min":>8} {"Max":>8} {"P95":>8}')
print('-' * 48)
for n, t in zip(names, timess):
    print(f'  {n:<15} {np.mean(t):>7.3f}ms {min(t):>7.3f}ms '
          f'{max(t):>7.3f}ms {np.percentile(t,95):>7.3f}ms')

## 2. Latency plots

In [ ]:
colors = ['#7F77DD', '#1D9E75', '#D85A30', '#BA7517']
avgs   = [np.mean(t) for t in timess]
stds   = [np.std(t)  for t in timess]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#f9f9f7')

bars = axes[0].bar(names, avgs, color=colors, width=0.5,
                    zorder=3, yerr=stds, capsize=4)
axes[0].set_title('Average latency per component (ms)', fontsize=11)
axes[0].set_ylabel('Latency (ms)')
axes[0].grid(axis='y', alpha=0.3, zorder=0)
for bar, val in zip(bars, avgs):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + max(stds)*0.05,
                 f'{val:.2f}ms', ha='center', fontsize=8)

bp = axes[1].boxplot(timess, patch_artist=True,
                      medianprops={'color': 'white', 'linewidth': 2})
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_xticklabels(names, fontsize=9)
axes[1].set_title('Latency distribution (100 trials)', fontsize=11)
axes[1].set_ylabel('Latency (ms)')
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Component Latency Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
os.makedirs('../results/plots', exist_ok=True)
plt.savefig('../results/plots/latency_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved')

## 3. Latency vs number of qubits

In [ ]:
qubit_counts = [500, 1000, 2000, 5000, 10000]
avg_times    = []

for n in qubit_counts:
    times = []
    for _ in range(20):
        t = time.perf_counter()
        BB84(n, channel).run()
        times.append((time.perf_counter() - t) * 1000)
    avg_times.append(np.mean(times))
    print(f'  {n} qubits: {np.mean(times):.2f}ms')

plt.figure(figsize=(9, 4))
plt.plot(qubit_counts, avg_times, 'o-', color='#7F77DD', linewidth=2)
plt.axhline(y=2000, color='#D85A30', linestyle='--', label='Target <2000ms')
plt.xlabel('Number of qubits')
plt.ylabel('Avg latency (ms)')
plt.title('BB84 latency vs qubit count')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../results/plots/latency_vs_qubits.png', dpi=150)
plt.show()

## Summary

In [ ]:
print('Latency Summary')
print('=' * 45)
print(f'BB84 QKD dominates latency: {np.mean(t_bb84):.2f}ms avg')
print(f'Kyber adds only:            {np.mean(t_kyber):.3f}ms')
print(f'HKDF adds only:             {np.mean(t_hkdf):.3f}ms')
print(f'Full pipeline target:       <2000ms')
print(f'Full pipeline achieved:     ~{np.mean(t_bb84)+np.mean(t_kyber)+np.mean(t_hkdf)+np.mean(t_enc):.2f}ms')
print(f'Conclusion: Hybrid overhead (Kyber+HKDF) is negligible vs QKD')